## Generación de numeros random aleatorios

En los papers GANs, DCGAN, redes adversariales de la revisión y estado del arte desarrollado, los mensajes binarios aleatorios son secuencias de bits m ∈ {0,1}^k generadas aleatoriamente en cada iteración del entrenamiento. No representan datos reales; su función es que la red aprenda el problema de cifrado de forma general, sin memorizarlo.
El proceso típico en estos papers normalmente sigue este flujo en cada mini-batch:
1. Generar mensaje aleatorio:  m ~ Uniforme({0,1}^k)
2. Generar clave aleatoria:    k ~ Uniforme({0,1}^n)
3. Transmisor (Alice) cifra:   c = Alice(m, k)
4. Receptor (Bob) descifra:    m̂ = Bob(c, k)
5. Atacante (Eva) intenta:     m̃ = Eve(c)
6. Calcular pérdidas y actualizar pesos

In [20]:
import torch
import torch.nn as nn

# Parámetros típicos en los papers, algunos han usado 32, pero no es lo normal. Se usan 16 bits para el mensaje y 16 bits para la clave.
# yo usaré solo 8 para mejorar visualización y comprensión de los resultados.

k = 8  # bits del mensaje
n = 8  # bits de la clave

# Red simple (solo para ver cómo entra el dato)
red = nn.Sequential(
    nn.Linear(k + n, 16),
    nn.ReLU(),
    nn.Linear(16, k),
    nn.Sigmoid()
)

# Generar un mensaje y una clave aleatorios
mensaje = torch.randint(0, 2, (k,)).float()  # ej: [1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0]
clave   = torch.randint(0, 2, (n,)).float()  # ej: [0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1]

# Concatenar antes de pasar a la red
entrada = torch.cat([mensaje, clave])        # tensor de 16 valores

# Forward pass
salida = red(entrada)
bits_salida = (salida > 0.5).int()
print("Mensaje:  ", mensaje)
print("Clave:    ", clave)
print("Entrada:  ", entrada)
print("Salida:   ", salida)
print("Salida (bits):  ", bits_salida)
print("Mensaje original:", mensaje.int())

Mensaje:   tensor([1., 0., 0., 0., 1., 1., 0., 1.])
Clave:     tensor([1., 1., 0., 0., 1., 0., 0., 0.])
Entrada:   tensor([1., 0., 0., 0., 1., 1., 0., 1., 1., 1., 0., 0., 1., 0., 0., 0.])
Salida:    tensor([0.5033, 0.4569, 0.5684, 0.4669, 0.5820, 0.5321, 0.5289, 0.5164],
       grad_fn=<SigmoidBackward0>)
Salida (bits):   tensor([1, 0, 1, 0, 1, 1, 1, 1], dtype=torch.int32)
Mensaje original: tensor([1, 0, 0, 0, 1, 1, 0, 1], dtype=torch.int32)


Ahora generaré un mensaje "hola mundo" como input.

In [ ]:
mensaje_texto = "hola mundo"
mensaje_binario = ''.join(format(ord(c), '08b') for c in mensaje_texto)
clave = torch.randint(0, 2, (16,)).float()
print("Mensaje en texto: ", mensaje_texto)
print("Mensaje en binario: ", mensaje_binario)
print("Clave aleatoria: ", clave)

Mensaje en texto:  hola mundo
Mensaje en binario:  01101000011011110110110001100001001000000110110101110101011011100110010001101111
Clave aleatoria:  tensor([1., 1., 0., 0., 1., 1., 0., 1., 1., 1., 1., 0., 1., 1., 0., 1.])


In [16]:
import torch
import torch.nn as nn

mensaje_texto_claro = "hola mundo"

# Convertir texto a bits
bits = [int(b) for c in mensaje_texto_claro for b in format(ord(c), '08b')]
mensaje = torch.tensor(bits).float()  # tensor de 80 bits

k = len(mensaje)  # k = 80
n = len(mensaje) # bits del mismo largo (solo por cumplir con premisa de Shanon)

red = nn.Sequential(
    nn.Linear(k + n, 64),
    nn.ReLU(),
    nn.Linear(64, k),
    nn.Sigmoid()
)

clave   = torch.randint(0, 2, (n,)).float()
entrada = torch.cat([mensaje, clave])

salida = red(entrada)
bits_salida = (salida > 0.5).int()

print("Mensaje (bits): ", mensaje)
print("Clave:          ", clave)
print("Salida:         ", salida)
print("Salida (bits):  ", bits_salida)
print("Mensaje original:", mensaje.int())

Mensaje (bits):  tensor([0., 1., 1., 0., 1., 0., 0., 0., 0., 1., 1., 0., 1., 1., 1., 1., 0., 1.,
        1., 0., 1., 1., 0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0.,
        0., 0., 0., 0., 0., 1., 1., 0., 1., 1., 0., 1., 0., 1., 1., 1., 0., 1.,
        0., 1., 0., 1., 1., 0., 1., 1., 1., 0., 0., 1., 1., 0., 0., 1., 0., 0.,
        0., 1., 1., 0., 1., 1., 1., 1.])
Clave:           tensor([1., 1., 1., 1., 0., 1., 1., 1., 1., 0., 1., 0., 1., 1., 1., 0., 1., 1.,
        0., 0., 1., 1., 0., 0., 1., 0., 0., 1., 1., 1., 0., 1., 0., 0., 0., 1.,
        1., 0., 1., 1., 1., 0., 1., 1., 0., 0., 1., 0., 1., 0., 1., 1., 0., 1.,
        1., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 0., 0., 1., 1., 1.])
Salida:          tensor([0.4474, 0.4603, 0.5038, 0.5283, 0.5604, 0.4933, 0.4476, 0.5778, 0.5596,
        0.4911, 0.5560, 0.5512, 0.5789, 0.4514, 0.4328, 0.5270, 0.5343, 0.4665,
        0.4845, 0.4684, 0.4705, 0.4469, 0.4846, 0.5082, 0.5363, 0.5